# FastAPI 에이전트 서버 호출 테스트

이 노트북은 **서버(`main.py`)가 실행 중** 이라는 전제로, 클라이언트 입장에서 API 를 호출한다.

먼저 다른 터미널에서 서버를 띄운다:
```bash
cd frameworks/langgraph/serving/fastapi
uv run uvicorn main:app --reload
```

그 뒤 아래 셀들을 실행한다. (서버가 켜져 있어야 한다)

## 1. `/ai-assist/invoke` — 동기 단발 호출
요청 한 번에 최종 답변을 JSON 으로 받는다.

In [ ]:
import requests

url = "http://127.0.0.1:8000/ai-assist/invoke"
payload = {"message": "https://langchain-ai.github.io/langgraph/agents/memory/ 내용을 한 문장으로 요약해줘."}

response = requests.post(url, json=payload)
print("Status:", response.status_code)
print("Response:", response.json()["content"])

## 2. `/ai-assist/stream` — SSE 스트리밍 호출
각 노드 업데이트가 실시간으로 흘러온다.

In [ ]:
import requests

url = "http://127.0.0.1:8000/ai-assist/stream"
params = {"message": "https://langchain-ai.github.io/langgraph/agents/memory/ 한 줄 요약, 한국어로!"}

with requests.get(url, params=params, stream=True) as response:
    print("Status:", response.status_code)
    for line in response.iter_lines(decode_unicode=True):
        if line:
            print(line)

## 정리

- LangGraph 그래프를 `graph.ainvoke` / `graph.astream` 으로 FastAPI 엔드포인트에 연결
- **invoke** = 최종 답변만 (동기), **stream** = 단계별 진행 (SSE)
- 노트북에서 실험하던 에이전트를 이렇게 **API 서버로 배포** 하면 외부 앱/프론트가 호출할 수 있다
- `uvicorn` 이 ASGI 서버로 async 엔드포인트를 구동, `--reload` 로 코드 변경 자동 반영
- `http://127.0.0.1:8000/docs` 에서 Swagger UI 로 직접 테스트도 가능